# Image Classification using CNN · Lab 8

Adapted from the supplied `image_classification_using_cnn.ipynb` course notebook. This version uses its shallow CNN and frozen InceptionV3 architectures and adds the Lab 8 exports and test-set projections.

Open this notebook in Google Colab and select a GPU runtime before running it. Both models use CIFAR-10, with a stratified validation split from the official training set and the official test set held out. The first run downloads CIFAR-10 and the ImageNet weights for InceptionV3.

Generated files go in `outputs/lab8`. Download the output archive before ending the Colab session.

In [ ]:
%pip install -q umap-learn

In [ ]:
import json
import shutil
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import tensorflow as tf
from IPython.display import JSON, display
from sklearn.manifold import TSNE
from sklearn.model_selection import train_test_split
from tensorflow import keras
from tensorflow.keras import layers
from umap import UMAP

SEED = 42
BATCH_SIZE = 32
OUTPUT = Path("outputs/lab8")
OUTPUT.mkdir(parents=True, exist_ok=True)
keras.utils.set_random_seed(SEED)

CLASS_NAMES = [
    "airplane", "automobile", "bird", "cat", "deer",
    "dog", "frog", "horse", "ship", "truck",
]
MODELS = {
    "small_cnn": {
        "epochs": 50,
        "optimizer": "sgd",
        "preprocessing": "32×32 RGB, scale pixel values from [0, 255] to [0, 1].",
        "initialization": "Random initialization",
    },
    "inception_v3": {
        "epochs": 10,
        "optimizer": "adam",
        "preprocessing": "32×32 RGB, bilinear resize to 128×128, scale pixels to [-1, 1].",
        "initialization": "Frozen ImageNet InceptionV3 with a new classification head",
    },
}

## 8.1 Train the CNN and InceptionV3

The small CNN learns its filters from scratch. InceptionV3 uses a frozen ImageNet backbone with a new CIFAR-10 classification head. The small CNN uses the course notebook’s single convolution and 100-unit dense layer. InceptionV3 uses its 1,024-unit head and a stratified 1,000-image subset of the official training split. Each model exposes the dense features before its softmax classifier. Preprocessing layers are part of each model so training and inference use the same pixel transformations.

References: [InceptionV3 preprocessing](https://www.tensorflow.org/api_docs/python/tf/keras/applications/InceptionV3), [Keras transfer learning guide](https://www.tensorflow.org/guide/keras/transfer_learning).

In [ ]:
(source_images, source_labels), (test_images, test_labels) = keras.datasets.cifar10.load_data()
source_labels = source_labels.ravel()
test_labels = test_labels.ravel()
train_ids, val_ids = train_test_split(
    np.arange(len(source_labels)), test_size=0.1, random_state=SEED, stratify=source_labels
)

def batches(images, labels, *, shuffle=False):
    dataset = tf.data.Dataset.from_tensor_slices((images, labels))
    if shuffle:
        dataset = dataset.shuffle(len(labels), seed=SEED)
    return dataset.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

def stratified_subset(indices, count):
    if len(indices) <= count:
        return indices
    selected, _ = train_test_split(
        indices, train_size=count, random_state=SEED, stratify=source_labels[indices]
    )
    return selected

model_splits = {
    "small_cnn": (train_ids, val_ids),
    "inception_v3": (stratified_subset(train_ids, 900), stratified_subset(val_ids, 100)),
}
test_data = batches(test_images, test_labels)

def build_model(name, *, backbone_weights=None):
    inputs = keras.Input(shape=(32, 32, 3), name="rgb_image")
    if name == "small_cnn":
        x = layers.Rescaling(1 / 255)(inputs)
        x = layers.Conv2D(32, 3, activation="relu", kernel_initializer="he_uniform")(x)
        x = layers.MaxPooling2D()(x)
        x = layers.Flatten()(x)
        features = layers.Dense(100, activation="relu", kernel_initializer="he_uniform", name="features")(x)
    elif name == "inception_v3":
        backbone = keras.applications.InceptionV3(
            include_top=False, weights=backbone_weights, input_shape=(128, 128, 3)
        )
        backbone.trainable = False
        x = layers.Resizing(128, 128, interpolation="bilinear")(inputs)
        x = layers.Rescaling(1 / 127.5, offset=-1)(x)
        x = backbone(x, training=False)
        x = layers.GlobalAveragePooling2D()(x)
        features = layers.Dense(1024, activation="relu", name="features")(x)
    else:
        raise ValueError(f"Unknown model: {name}")
    predictions = layers.Dense(len(CLASS_NAMES), activation="softmax")(features)
    return keras.Model(inputs, predictions, name=name)

In [ ]:
for name, config in MODELS.items():
    model_train_ids, model_val_ids = model_splits[name]
    train_data = batches(source_images[model_train_ids], source_labels[model_train_ids], shuffle=True)
    val_data = batches(source_images[model_val_ids], source_labels[model_val_ids])
    model = build_model(name, backbone_weights="imagenet" if name == "inception_v3" else None)
    model.compile(
        optimizer=config["optimizer"],
        loss=keras.losses.SparseCategoricalCrossentropy(),
        metrics=["accuracy"],
    )
    history = model.fit(train_data, validation_data=val_data, epochs=config["epochs"], shuffle=False)
    scores = model.evaluate(test_data, return_dict=True)
    model.save_weights(OUTPUT / f"{name}.weights.h5")
    model.save(OUTPUT / f"{name}.keras")

    passport = {
        "model": name,
        "dataset": "CIFAR-10",
        "sample_counts": {
            "train": len(model_train_ids),
            "validation": len(model_val_ids),
            "test": len(test_labels),
        },
        "class_names": CLASS_NAMES,
        "external_image_preprocessing": "Convert to RGB and resize to 32×32 with bilinear interpolation.",
        "model_preprocessing": config["preprocessing"],
        "initialization": config["initialization"],
        "seed": SEED,
        "batch_size": BATCH_SIZE,
        "optimizer": config["optimizer"],
        "training_history": {key: [float(x) for x in values] for key, values in history.history.items()},
        "test_metrics": {key: float(value) for key, value in scores.items()},
    }
    (OUTPUT / f"{name}.passport.json").write_text(
        json.dumps(passport, indent=2, ensure_ascii=False), encoding="utf-8"
    )

## 8.2 Load saved weights and classify an image

Choose either trained model. Set `IMAGE_PATH` to an image uploaded through Colab’s Files panel, or leave it empty to classify an official test image. Images should depict one of the ten CIFAR-10 classes.

In [ ]:
MODEL_NAME = "inception_v3"  # "small_cnn" or "inception_v3"
IMAGE_PATH = ""
TEST_INDEX = 0

passport = json.loads((OUTPUT / f"{MODEL_NAME}.passport.json").read_text(encoding="utf-8"))
display(JSON(passport))
loaded_model = build_model(passport["model"])
loaded_model.load_weights(OUTPUT / f"{MODEL_NAME}.weights.h5")

if IMAGE_PATH:
    image = keras.utils.img_to_array(
        keras.utils.load_img(IMAGE_PATH, color_mode="rgb", target_size=(32, 32), interpolation="bilinear")
    )
else:
    image = test_images[TEST_INDEX]
    print(f"True class: {CLASS_NAMES[test_labels[TEST_INDEX]]}")

probabilities = loaded_model.predict(image[np.newaxis], verbose=0)[0]
best = int(np.argmax(probabilities))
prediction = passport["class_names"][best]
plt.imshow(image.astype(np.uint8))
plt.title(f"{prediction} ({probabilities[best]:.1%})")
plt.axis("off")
plt.show()

## 8.3 Save, reload and project test features

Extract one penultimate-layer feature vector per official test image from the selected model. The next cell reads the saved vectors and projects the same samples with t-SNE and UMAP. Point colors show ground-truth classes.

In [ ]:
extractor = keras.Model(loaded_model.input, loaded_model.get_layer("features").output)
features = extractor.predict(test_data.map(lambda images, labels: images), verbose=1)
feature_path = OUTPUT / f"{MODEL_NAME}.test_features.npz"
np.savez_compressed(
    feature_path, features=features, labels=test_labels, class_names=passport["class_names"]
)
print(f"Saved {features.shape[0]} vectors of length {features.shape[1]} to {feature_path}")

In [ ]:
with np.load(feature_path) as saved:
    features = saved["features"]
    labels = saved["labels"]
    names = saved["class_names"]

methods = {
    "t-SNE": TSNE(n_components=2, perplexity=30, init="pca", random_state=SEED),
    "UMAP": UMAP(n_components=2, n_neighbors=15, random_state=SEED, n_jobs=1),
}
fig, axes = plt.subplots(1, 2, figsize=(13, 5), layout="constrained")
for ax, (name, method) in zip(axes, methods.items()):
    coordinates = method.fit_transform(features)
    for label, class_name in enumerate(names):
        points = coordinates[labels == label]
        ax.scatter(points[:, 0], points[:, 1], s=6, alpha=0.55, label=class_name)
    ax.set_title(name)
    ax.set(xticks=[], yticks=[])
axes[-1].legend(title="True class", loc="upper left", bbox_to_anchor=(1.02, 1))
fig.savefig(OUTPUT / f"{MODEL_NAME}.projections.png", dpi=180, bbox_inches="tight")
plt.show()

## Download the results

The archive includes both models, their weights and passports, plus the selected model’s test features and projection figure. Extract it and select a `.keras` or `.test_features.npz` file in the corresponding ForensicsApp Lab 8 tool.

In [ ]:
from google.colab import files

archive = shutil.make_archive(str(OUTPUT), "zip", root_dir=OUTPUT)
files.download(archive)